# Sequence-to-Sequence ও Attention

## এই notebook সম্পর্কে

এটি `example.py`-এর হুবহু কোড, notebook-এ বিভক্ত। দুটি from-scratch NumPy demo:

1. Attention একটি *soft, differentiable lookup* হিসেবে — একটি query দেওয়া হলে, প্রতিটি key কতটা
   মেলে তার ভিত্তিতে value-গুলোর weighted blend retrieve হয়।
2. Sequence দীর্ঘ হওয়ার সাথে সাথে Seq2Seq bottleneck-এর (শুধু final encoder hidden state-এর উপর
   নির্ভর করা) বনাম attention-এর (প্রতিটি encoder hidden state থেকে retrieve করা) সরাসরি তুলনা।

**চালানো:** মূল ফোল্ডারে `python example.py`, অথবা এই notebook-এর cell-গুলো ক্রমান্বয়ে চালান।

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

## ১. হেল্পার ফাংশন: softmax, cosine_similarity ও attention

`softmax` score-গুলোকে probability-সদৃশ weight-এ রূপান্তর করে। `attention` dot-product
score দিয়ে weight বানিয়ে value-গুলোর weighted sum (context vector) ফেরত দেয় —
সাথে weight-গুলোও, যাতে দেখা যায় কোন position কতটা গুরুত্ব পেয়েছে।

In [ ]:
def softmax(scores):
    shifted = scores - np.max(scores)
    exps = np.exp(shifted)
    return exps / exps.sum()


def cosine_similarity(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


def attention(query, keys, values):
    """Dot-product attention: query (D,), keys (N, D), values (N, D_v)।
    ফেরত দেয় (context vector, attention weights)।"""
    scores = keys @ query                      # (N,) -- প্রতি key-এ একটি score
    weights = softmax(scores)                   # (N,) -- যোগফল 1
    context = weights @ values                   # value-গুলোর weighted sum
    return context, weights

## ২. Demo 1: attention একটি soft lookup হিসেবে

6টি key/value জোড়া তৈরি হয়। তারপর key[3]-এর একটি noisy কপিকে query করে দেখা হয়,
attention ঠিক কোন value retrieve করে — এটিই 'content দিয়ে মেলানো, নির্দিষ্ট position নয়'
আচরণ, যার কারণে attention একটি নির্দিষ্ট slot-এর চেয়ে অনেক ভালো generalize করে।

In [ ]:
def soft_lookup_demo():
    print("=" * 70)
    print("1. ATTENTION AS A SOFT, DIFFERENTIABLE LOOKUP")
    print("=" * 70)

    num_items = 6
    dim = 16
    keys = rng.normal(size=(num_items, dim))
    # Values হলো প্রতি key-এর জন্য আলাদা, স্পষ্টভাবে লেবেল করা payload, যাতে
    # ঠিক কোন value retrieve হলো তা বোঝা যায়।
    values = np.eye(num_items, dim)  # value i হলো (মোটামুটি) one-hot-এর মতো row i

    target_idx = 3
    query = keys[target_idx] + rng.normal(scale=0.1, size=dim)  # key[3]-এর একটি noisy কপি

    context, weights = attention(query, keys, values)

    print(f"Query = a noisy copy of key[{target_idx}]")
    print("Attention weights over the 6 keys:")
    for i, w in enumerate(weights):
        marker = "  <-- target" if i == target_idx else ""
        print(f"  key[{i}]: {w:.4f}{marker}")

    sim_to_target_value = cosine_similarity(context, values[target_idx])
    print(f"\ncosine(context, value[{target_idx}]) = {sim_to_target_value:.4f}")
    print("-> Even though the query was never exactly equal to any key, attention")
    print("   correctly retrieves (almost) the right value by content similarity --")
    print("   this 'retrieve by matching content, not by fixed position' behavior")
    print("   is exactly why attention generalizes so much better than a fixed slot.")


soft_lookup_demo()

## ৩. Demo 2: Seq2Seq bottleneck বনাম attention

উৎস sequence-এর position 0-তে একটি গুরুত্বপূর্ণ 'signal' থাকে। দুটি উপায়ে বানানো
'context vector' থেকে সেই প্রথম position-এর তথ্য কতটা উদ্ধার করা যায়, তা sequence
দীর্ঘ হওয়ার সাথে সাথে তুলনা করা হয় — bottleneck (শুধু h_last) বনাম attention
(সব hidden state থেকেই retrieve)।

In [ ]:
def encode_all_hidden_states(x_seq, Wxh, Whh, bh, hidden_dim):
    """একটি ন্যূনতম vanilla-RNN encoder যা প্রতিটি hidden state ফেরত দেয়,
    শুধু শেষটি নয় — attention সম্ভব হওয়ার জন্য এটিই প্রয়োজন।"""
    h = np.zeros(hidden_dim)
    hidden_states = []
    for x in x_seq:
        h = np.tanh(Wxh @ x + Whh @ h + bh)
        hidden_states.append(h)
    return np.array(hidden_states)  # shape (T, hidden_dim)


def bottleneck_vs_attention_demo():
    print("\n" + "=" * 70)
    print("2. FIXED-CONTEXT BOTTLENECK vs. ATTENTION, AS SEQUENCE LENGTH GROWS")
    print("=" * 70)
    print("Setup: an important 'signal' occurs at position 0 of the source")
    print("sequence. We ask two different ways of building a 'context vector'")
    print("to recover information about that first position, as the sentence")
    print("(sequence length T) gets longer.\n")

    input_dim, hidden_dim = 8, 16
    scale = 1.0 / np.sqrt(hidden_dim)
    Wxh = rng.normal(0, scale, size=(hidden_dim, input_dim))
    Whh = rng.normal(0, scale, size=(hidden_dim, hidden_dim))
    bh = np.zeros(hidden_dim)

    max_len = 40
    signal = rng.normal(size=input_dim)
    noise_tokens = [rng.normal(size=input_dim) for _ in range(max_len - 1)]
    full_sequence = [signal] + noise_tokens

    print(f"{'seq_len':>8}  {'fixed-context sim':>19}  {'attention sim':>15}")
    for T in [2, 5, 10, 20, 30, 40]:
        x_seq = full_sequence[:T]
        hidden_states = encode_all_hidden_states(x_seq, Wxh, Whh, bh, hidden_dim)
        h_first, h_last = hidden_states[0], hidden_states[-1]

        # Fixed-context (ক্লাসিক Seq2Seq): decoder-এর কাছে শুধু h_last টিকে থাকে।
        fixed_context_sim = cosine_similarity(h_last, h_first)

        # Attention: decoder সব hidden state-কে query করতে পারে, h_first-সহ।
        # h_first-কে query হিসেবে ব্যবহার "decoder এমন একটি প্রশ্ন করে যার উত্তর
        # position 0-তে থাকে" -এর stand-in — dot-product attention স্বাভাবিকভাবেই
        # একটি vector-কে নিজের সাথে সবচেয়ে বেশি score দেয়।
        context, _ = attention(query=h_first, keys=hidden_states, values=hidden_states)
        attention_sim = cosine_similarity(context, h_first)

        print(f"{T:>8}  {fixed_context_sim:>19.4f}  {attention_sim:>15.4f}")

    print("\n-> Fixed-context similarity is low and noisy even for short sequences,")
    print("   and never recovers: with an untrained (random-weight) RNN, each new")
    print("   timestep's tanh recurrence mixes in enough new information that the")
    print("   final hidden state stops resembling h_first almost immediately --")
    print("   exactly the bottleneck problem described in the README, just visible")
    print("   even faster here than it would be in a fully trained network.")
    print("-> Attention similarity stays pinned near 1.0 regardless of T, because")
    print("   the decoder can always look directly back at h_first instead of")
    print("   relying on whatever survived being compressed into one final vector.")


bottleneck_vs_attention_demo()

## ৪. main()
`main()` দুটি demo-ই চালায় — উপরের cell-গুলোতে প্রতিটি demo তার সংজ্ঞার পরেই
ইতিমধ্যে একবার চালানো হয়েছে; `main()` এগুলোর পূর্ণ-চলমান সমতুল্য।

In [ ]:
def main():
    soft_lookup_demo()
    bottleneck_vs_attention_demo()

In [ ]:
main()